# 0v · **영상 저장** — eval mp4

우리 파이프라인의 clean-eval 은 **영상을 끄고** 돈다 (`lerobot_eval` 이 속도/OOM 때문에
`max_episodes_rendered=0` 하드코딩). 그래서 SR/떨림은 나와도 mp4 는 안 생긴다.

이 노트북은 `RECORD_VIDEOS` 환경변수로 렌더를 켜서 **몇 개 에피소드만 mp4 로** 저장한다.
SR/떨림 결과와는 **별도 폴더**(`.../seed<N>/videos_rep/`)에 저장 → 기존 결과를 안 건드림.

> 영상은 무겁고 느리다. 대표 seed 하나 · 5개 정도만 뽑아 데모용으로.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = cf.SHORT_SIM       # 'transfer'  (libero 는 cf.SUPPORT_SIM, insertion 은 cf.MAIN_SIM)
TAGS   = ['ours', 'acm']    # 영상 뽑을 모델
SEED   = 0                  # 영상은 대표 seed 하나면 충분
N_VID  = 5                  # 저장할 에피소드 mp4 수
GPUS   = cf.v23.available_gpus()

print('task:', TASK, '| 모델:', TAGS, '| seed:', SEED, '| 영상', N_VID, '개씩')
print('GPU :', GPUS)

## 1) 체크포인트 확인


In [ ]:
# 150k 체크포인트 있는지
ok = cf.print_ckpt_status([t for t in TAGS if t != 'act_te'], [SEED], TASK)
print('\n=>', '진행 가능' if ok else '⚠️ 학습 먼저')

## 2) 영상 저장 실행


In [ ]:
# 모델당 영상 eval 하나씩 (GPU 나눠서)
labeled = []
for i, t in enumerate(TAGS):
    g = GPUS[i % len(GPUS)]
    try:
        cmd = cf.record_videos_cmd(t, SEED, task=TASK, gpu_id=g, n_videos=N_VID)
        labeled.append((f'{t}/seed{SEED}/videos', cmd))
    except FileNotFoundError as e:
        print('skip:', e)

for j in range(0, len(labeled), len(GPUS)):
    cf.launch_cmds_live(labeled[j:j + len(GPUS)])
print('\n영상 저장 완료')

## 3) 저장된 mp4 목록


In [ ]:
# 저장된 mp4 확인
import glob
for t in TAGS:
    vdir = cf.v23.eval_clean_dir(t, SEED, TASK) / 'videos_rep'
    mp4 = sorted(glob.glob(str(vdir / '**' / '*.mp4'), recursive=True))
    print(f'{t}: {len(mp4)}개')
    for m in mp4:
        print('   ', Path(m).relative_to(vdir), f"({Path(m).stat().st_size/1024:.0f} KB)")

## 4) 영상만 zip → 팀 전송


In [ ]:
# 영상만 zip 으로 → 팀에 전송
import shutil
stage = cf.OUTPUT_BASE / 'share' / f'{TASK}_videos'
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True)
for t in TAGS:
    vdir = cf.v23.eval_clean_dir(t, SEED, TASK) / 'videos_rep'
    for i, m in enumerate(sorted(vdir.rglob('*.mp4'))):
        shutil.copy(m, stage / f'{t}_seed{SEED}_ep{i}.mp4')
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'{TASK}_videos'), 'zip', root_dir=stage)
print('보낼 파일:', zip_path)
for p in sorted(stage.glob('*.mp4')):
    print('  ', p.name, f'({p.stat().st_size/1024:.0f} KB)')